# 04 — Gold Business Metrics

**Project:** Incremental Orders Pipeline with Delta Lake  
**Layer:** Gold  
**Source:** `orders_project.silver_orders_current`  
**Targets:**

- `orders_project.gold_orders_by_status`
- `orders_project.gold_daily_sales`
- `orders_project.gold_order_summary`

This notebook creates business-ready Gold tables from the Silver current-state orders table.

The Gold layer answers business questions such as:

- How many current orders exist by status?
- How much revenue has been recognized from delivered orders?
- How much order amount is still pending or in progress?
- What is the cancellation rate?
- What is the total value of current orders?

In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    to_date
)

In [0]:
spark.sql("USE SCHEMA orders_project")

In [0]:
silver_df = spark.table("orders_project.silver_orders_current")

display(
    silver_df.select(
        "order_id",
        "customer_id",
        "order_status",
        "order_amount",
        "last_event_ts",
        "last_batch_id"
    ).orderBy("order_id")
)

In [0]:
silver_df.createOrReplaceTempView("silver_orders_current_temp")

print("Temporary view created: silver_orders_current_temp")

In [0]:
gold_orders_by_status_df = spark.sql("""
SELECT
    order_status,
    COUNT(*) AS total_orders,
    ROUND(SUM(order_amount), 2) AS total_amount_usd,
    ROUND(AVG(order_amount), 2) AS avg_order_amount_usd,
    MIN(last_event_ts) AS first_event_ts,
    MAX(last_event_ts) AS latest_event_ts,
    current_timestamp() AS gold_processed_ts
FROM silver_orders_current_temp
GROUP BY order_status
ORDER BY total_orders DESC
""")

display(gold_orders_by_status_df)

In [0]:
gold_orders_by_status_table = "orders_project.gold_orders_by_status"

gold_orders_by_status_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_orders_by_status_table)

print(f"Gold table created: {gold_orders_by_status_table}")

In [0]:
gold_daily_sales_df = spark.sql("""
SELECT
    TO_DATE(last_event_ts) AS order_activity_date,

    COUNT(*) AS total_current_orders,

    ROUND(SUM(order_amount), 2) AS gross_current_order_amount_usd,

    ROUND(SUM(
        CASE 
            WHEN order_status = 'delivered' THEN order_amount 
            ELSE 0 
        END
    ), 2) AS recognized_revenue_usd,

    ROUND(SUM(
        CASE 
            WHEN order_status = 'cancelled' THEN order_amount 
            ELSE 0 
        END
    ), 2) AS cancelled_amount_usd,

    SUM(CASE WHEN order_status = 'pending' THEN 1 ELSE 0 END) AS pending_orders,
    SUM(CASE WHEN order_status = 'shipped' THEN 1 ELSE 0 END) AS shipped_orders,
    SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
    SUM(CASE WHEN order_status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,

    current_timestamp() AS gold_processed_ts

FROM silver_orders_current_temp
GROUP BY TO_DATE(last_event_ts)
ORDER BY order_activity_date
""")

display(gold_daily_sales_df)

In [0]:
gold_daily_sales_table = "orders_project.gold_daily_sales"

gold_daily_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_daily_sales_table)

print(f"Gold table created: {gold_daily_sales_table}")

In [0]:
gold_order_summary_df = spark.sql("""
SELECT
    COUNT(*) AS total_current_orders,
    COUNT(DISTINCT customer_id) AS total_customers,

    ROUND(SUM(order_amount), 2) AS gross_current_order_amount_usd,

    ROUND(SUM(
        CASE 
            WHEN order_status = 'delivered' THEN order_amount 
            ELSE 0 
        END
    ), 2) AS recognized_revenue_usd,

    ROUND(SUM(
        CASE 
            WHEN order_status != 'cancelled' THEN order_amount 
            ELSE 0 
        END
    ), 2) AS active_order_amount_usd,

    ROUND(SUM(
        CASE 
            WHEN order_status = 'cancelled' THEN order_amount 
            ELSE 0 
        END
    ), 2) AS cancelled_amount_usd,

    SUM(CASE WHEN order_status = 'pending' THEN 1 ELSE 0 END) AS pending_orders,
    SUM(CASE WHEN order_status = 'shipped' THEN 1 ELSE 0 END) AS shipped_orders,
    SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered_orders,
    SUM(CASE WHEN order_status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders,

    ROUND(AVG(order_amount), 2) AS avg_order_value_usd,

    ROUND(
        SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) / COUNT(*) * 100,
        2
    ) AS delivered_rate_pct,

    ROUND(
        SUM(CASE WHEN order_status = 'cancelled' THEN 1 ELSE 0 END) / COUNT(*) * 100,
        2
    ) AS cancellation_rate_pct,

    current_timestamp() AS gold_processed_ts

FROM silver_orders_current_temp
""")

display(gold_order_summary_df)

In [0]:
gold_order_summary_table = "orders_project.gold_order_summary"

gold_order_summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_order_summary_table)

print(f"Gold table created: {gold_order_summary_table}")

In [0]:
display(
    spark.sql("""
    SHOW TABLES IN orders_project
    """)
)

In [0]:
display(spark.sql("""
SELECT 'bronze_orders_raw' AS table_name, COUNT(*) AS total_records
FROM orders_project.bronze_orders_raw

UNION ALL

SELECT 'silver_orders_current' AS table_name, COUNT(*) AS total_records
FROM orders_project.silver_orders_current

UNION ALL

SELECT 'gold_orders_by_status' AS table_name, COUNT(*) AS total_records
FROM orders_project.gold_orders_by_status

UNION ALL

SELECT 'gold_daily_sales' AS table_name, COUNT(*) AS total_records
FROM orders_project.gold_daily_sales

UNION ALL

SELECT 'gold_order_summary' AS table_name, COUNT(*) AS total_records
FROM orders_project.gold_order_summary
"""))

In [0]:
display(
    spark.sql("""
    SELECT
        order_id,
        customer_id,
        order_status,
        order_amount,
        last_event_ts,
        last_batch_id
    FROM orders_project.silver_orders_current
    ORDER BY order_id
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM orders_project.gold_orders_by_status
    ORDER BY total_orders DESC
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM orders_project.gold_daily_sales
    ORDER BY order_activity_date
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM orders_project.gold_order_summary
    """)
)

In [0]:
display(
    spark.sql("""
    DESCRIBE HISTORY orders_project.gold_order_summary
    """).select(
        "version",
        "timestamp",
        "operation",
        "operationParameters",
        "operationMetrics"
    )
)